### 1. Preparación de Databricks: cargar los archivos

# Paso 1. Crear el esquema y el volumen

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electiva;

CREATE VOLUME IF NOT EXISTS electiva.datos;


### Paso 2. Subir los CSV

### Paso 3. Crear las tablas Delta

In [0]:
archivos = {
    "customers": "customers.csv",
    "employee": "employees.csv",
    "orders": "orders.csv",
    "product_categories": "productcategories.csv",
    "product_sub_categories": "productsubcategories.csv",
    "products": "products.csv",
    "vendor": "vendors.csv",
    "vendor_product": "vendorproduct.csv"
}

ruta = "/Volumes/workspace/electiva/datos/"

for tabla, archivo in archivos.items():

    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .option("encoding", "ISO-8859-1")
          .csv(ruta + archivo))

    (df.write
       .format("delta")
       .mode("overwrite")
       .saveAsTable(f"electiva.{tabla}"))

    print(f"Tabla creada: {tabla}")

In [0]:
%sql
SHOW TABLES IN electiva;

# 2. Desarrollo del parcial

### Punto 1. Crear fullname y user_name (0.5)

In [0]:
%sql

SELECT
    CustomerID AS customerid,

    CONCAT(FirstName, ' ', LastName) AS fullname,

    CONCAT(
        CAST(CustomerID AS STRING),
        '-',
        UPPER(SUBSTRING(FirstName, 1, 2)),
        '-',
        LOWER(SUBSTRING(LastName, 1, 2)),
        '-AW'
    ) AS user_name

FROM electiva.customers;


### punto 2. crear la tabla page_user


In [0]:
%sql

CREATE TABLE IF NOT EXISTS electiva.page_user (
    customerid INT,
    user_name STRING
)
USING DELTA;


In [0]:
%sql

INSERT OVERWRITE TABLE electiva.page_user

SELECT
    CustomerID AS customerid,

    CONCAT(
        CAST(CustomerID AS STRING),
        '-',
        UPPER(SUBSTRING(FirstName, 1, 2)),
        '-',
        LOWER(SUBSTRING(LastName, 1, 2)),
        '-AW'
    ) AS user_name

FROM electiva.customers;


In [0]:
%sql
SELECT *
FROM electiva.page_user;

### Punto 3. Descuentos por categoría (1.5)

In [0]:
%sql

CREATE OR REPLACE VIEW electiva.orders_august_2011 AS

WITH ventas AS (

    SELECT
        p.ProductName AS productname,
        p.ModelName AS modelname,
        pu.user_name,
        o.SalesOrderID AS salesorderid,
        pc.CategoryName AS categoryname,

        CAST(o.OrderDate AS DATE) AS orderdate,

        o.LineTotal AS linetotal,

        CASE
            WHEN pc.CategoryName = 'Bikes' THEN 0.10
            WHEN pc.CategoryName = 'Clothing' THEN 0.15
            WHEN pc.CategoryName = 'Components' THEN 0.08
            WHEN pc.CategoryName = 'Accessories' THEN 0.20
            ELSE 0
        END AS `descuento_mitad_año`

    FROM electiva.orders o

    INNER JOIN electiva.products p
        ON o.ProductID = p.ProductID

    INNER JOIN electiva.product_sub_categories psc
        ON p.SubCategoryID = psc.SubCategoryID

    INNER JOIN electiva.product_categories pc
        ON psc.CategoryID = pc.CategoryID

    INNER JOIN electiva.page_user pu
        ON o.CustomerID = pu.customerid
)

SELECT
    productname,
    modelname,
    user_name,
    salesorderid,
    categoryname,
    orderdate,
    linetotal,
    `descuento_mitad_año`,

    linetotal - (linetotal * `descuento_mitad_año`)
    AS total_descuentos_aplicados

FROM ventas

WHERE orderdate >= DATE '2011-08-01'
  AND orderdate < DATE '2011-09-01';


In [0]:
%sql
SELECT *
FROM electiva.orders_august_2011;

In [0]:
%sql
SELECT
    OrderDate,
    CAST(OrderDate AS DATE) AS fecha_convertida
FROM electiva.orders
LIMIT 10;

In [0]:
%sql
SELECT
    COALESCE(
        TRY_TO_DATE(CAST(OrderDate AS STRING), 'M/d/yyyy'),
        TRY_TO_DATE(CAST(OrderDate AS STRING), 'yyyy-MM-dd')
    ) AS orderdate
FROM electiva.orders;

In [0]:
%sql
SELECT
    CAST(OrderDate AS DATE) AS orderdate
FROM electiva.orders;


### Punto 4. Clasificación de clientes (2.0)

In [0]:
%sql

CREATE OR REPLACE MATERIALIZED VIEW electiva.customer_class AS

WITH ordenes_clientes AS (

    SELECT
        CustomerID,
        COUNT(DISTINCT SalesOrderID) AS amount_of_orders

    FROM electiva.orders

    WHERE YEAR(
        COALESCE(
            TRY_TO_DATE(CAST(OrderDate AS STRING), 'M/d/yyyy'),
            TRY_TO_DATE(CAST(OrderDate AS STRING), 'yyyy-MM-dd')
        )
    ) = 2011

    GROUP BY CustomerID

),

clasificacion AS (

    SELECT
        c.CustomerID AS customerid,
        c.FullName AS customer_fullname,
        pu.user_name,
        COALESCE(oc.amount_of_orders, 0) AS amount_of_orders,

        CASE
            WHEN COALESCE(oc.amount_of_orders, 0) = 3
                THEN 'Oro'

            WHEN COALESCE(oc.amount_of_orders, 0) = 2
                THEN 'Plata'

            WHEN COALESCE(oc.amount_of_orders, 0) < 2
                THEN 'Ocasional'

            ELSE 'Sin clasificar'
        END AS user_level

    FROM electiva.customers c

    INNER JOIN electiva.page_user pu
        ON c.CustomerID = pu.customerid

    LEFT JOIN ordenes_clientes oc
        ON c.CustomerID = oc.CustomerID

)

SELECT
    amount_of_orders,
    user_name,
    customer_fullname,
    user_level,
    customerid

FROM clasificacion

WHERE amount_of_orders > 0;


In [0]:
%sql
SELECT *
FROM electiva.customer_class;

In [0]:
%sql
-- Punto 1
SELECT * FROM electiva.customers LIMIT 10;

-- Punto 2
SELECT * FROM electiva.page_user LIMIT 10;

-- Punto 3
SELECT * FROM electiva.orders_august_2011 LIMIT 20;

-- Punto 4
SELECT * FROM electiva.customer_class LIMIT 20;


In [0]:
%sql
SELECT COUNT(*) FROM electiva.page_user;

SELECT COUNT(*) FROM electiva.orders_august_2011;

SELECT COUNT(*) FROM electiva.customer_class;